# Full Pipeline Launcher — Stage 1 + Stage 2 + Stage 3

This notebook runs the **streamlined three-stage retrain** (the May-2026 v2.0 recipe) end-to-end via `scripts/training/train_pipeline.py`. It is the entry point you'll use whenever you have new news articles to learn from.

**What happens:**
1. **Stage 1** (~3-4 hr on G4 95GB) — NER-only training with per-sample global-attention dropout (p=0.3, warmup 30%).
2. **Stage 2** (~3-4 hr) — Joint NER + sentiment with curriculum, continuing dropout.
3. **Stage 3** (~3-4 hr) — Fresh SentimentHead retrain on FROZEN Stage 1 backbone. Encoder is locked, V2 sentiment head trains from Xavier init.
4. **Auto-package** the best Stage 3 epoch into `trained_model/v<VERSION>_<DATE>/` (model.pt, config.json, tokenizer, MODEL_CARD.md).

**Total wall-clock**: ~12-14 hours on a 95 GB GPU. Fits in a single Colab Pro+ session.

**Why Stage 3 loads from Stage 1 (not Stage 2):** Stage 2's joint training slightly degrades CLS-only NER F1 (~0.58 → 0.53). Stage 3 discards the sentiment head from its input anyway and freezes the encoder, so we use the stronger Stage 1 backbone for better e2e NER.

## When to use this notebook
- Gathered more news articles → update `data/labeled/final/train.jsonl`, run this.
- Switched encoder (e.g., to a larger Longformer variant) → override `--encoder-name` and rerun.
- Adjusted hyperparameters → override the flags in Cell 4.

## If a stage crashes mid-run
- Intermediate checkpoints land on `/content/stage{1,2,3}_<run_id>/`.
- Re-run Cell 4 with `--skip-stages 1,2` to resume from Stage 3 (or similar).
- Set `--run-id` to a fixed string to make resumption deterministic.

In [ ]:
# 1. Mount Drive & verify GPU (G4 95 GB recommended)
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Set project path and run parameters
import os
from datetime import datetime

PROJECT_PATH = "/content/drive/MyDrive/entity_sentiment_model_pipeline"
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
VERSION = "v3.0"
OUTPUT_VERSION_NAME = f"{VERSION}_{datetime.now().strftime('%Y%m%d')}"

assert os.path.exists(PROJECT_PATH), f"Not found: {PROJECT_PATH}"
assert os.path.exists(f"{PROJECT_PATH}/scripts/training/train_pipeline.py"), "train_pipeline.py not found!"
assert os.path.exists(f"{PROJECT_PATH}/data/labeled/final/train.jsonl"), "training data not found!"

print(f"Project          : {PROJECT_PATH}")
print(f"Run ID           : {RUN_ID}")
print(f"Final model dir  : trained_model/{OUTPUT_VERSION_NAME}/")

In [ ]:
# 3. Install dependencies
!pip install -q transformers torch torchvision torchaudio
!pip install -q pytorch-crf

In [ ]:
# 4. Run the full pipeline (Stage 1 -> Stage 2 -> Stage 3 -> package)
#
# Defaults are the May-2026 v2.0 recipe. Override below if you've changed something:
#   --stage1-epochs, --stage2-epochs, --stage3-epochs
#   --batch-size-train (Stages 1+2),  --stage3-batch (Stage 3)
#   --global-attn-dropout-prob 0.3 (the proven default)
#
# To resume after a crash, add e.g. --skip-stages 1,2 and use the same --run-id
# as the original run to reuse intermediate checkpoints.

!cd {PROJECT_PATH} && python scripts/training/train_pipeline.py \
    --run-id {RUN_ID} \
    --version {VERSION} \
    --output-version-name {OUTPUT_VERSION_NAME} \
    --local-ckpt-root /content \
    --drive-ckpt-root {PROJECT_PATH}/checkpoints

In [ ]:
# 5. Verify the final bundle
import os
bundle = f"{PROJECT_PATH}/trained_model/{OUTPUT_VERSION_NAME}"
print(f"Final bundle: {bundle}")
if os.path.exists(bundle):
    for f in sorted(os.listdir(bundle)):
        path = os.path.join(bundle, f)
        if os.path.isfile(path):
            print(f"  {f:30s} {os.path.getsize(path) / 1e6:8.1f} MB")
        else:
            print(f"  {f}/")
    print()
    print("Model card:")
    with open(os.path.join(bundle, "MODEL_CARD.md")) as fh:
        for line in fh.readlines()[:30]:
            print("  " + line.rstrip())
else:
    print("  Bundle not built — check the log above for the failing stage.")

In [ ]:
# 6. (Optional) Run the e2e evaluation on the freshly trained model
#    Estimated time: ~4-6 minutes on G4 with batched single-pass.

CHECKPOINT_PATH = f"{PROJECT_PATH}/trained_model/{OUTPUT_VERSION_NAME}/model.pt"

!cd {PROJECT_PATH} && python scripts/evaluation/evaluate_e2e_pipeline.py \
    --checkpoint {CHECKPOINT_PATH} \
    --benchmark {PROJECT_PATH}/data/labeled/final/holdout_relabeled.jsonl \
    --output-dir {PROJECT_PATH}/outputs/e2e_evaluation \
    --local-output-dir /content \
    --ner-mode single-pass \
    --inference-batch-size 16 \
    --iou-threshold 0.5 \
    --max-length 2048 \
    --log-every 100

In [ ]:
# 7. Terminate runtime when done (stop billing)
from google.colab import drive, runtime
try:
    drive.flush_and_unmount()
except Exception as e:
    print(f'flush_and_unmount: {e}')
runtime.unassign()